<a href="https://colab.research.google.com/github/dltkdgk0409-coder/Maritime_Data_Mining/blob/main/%ED%95%B4%EC%82%AC%EB%8D%B0%EC%9D%B4%ED%84%B0%EB%A7%88%EC%9D%B4%EB%8B%9D_13%EC%A3%BC%EC%B0%A8_%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
# CSV 파일 없을 때 샘플 데이터 직접 생성
import os
import pandas as pd
import numpy as np

os.makedirs('/content/type3', exist_ok=True)

# churn_logit1.csv 생성
np.random.seed(42)
n = 500
df_churn = pd.DataFrame({
    'age': np.random.randint(20, 60, n),
    'usage_hour': np.random.randint(1, 100, n),
    'complaint_cnt': np.random.randint(0, 10, n),
})
log_odds = -0.0234*df_churn['age'] - 0.0384*df_churn['usage_hour'] + 0.6561*df_churn['complaint_cnt']
prob = 1 / (1 + np.exp(-log_odds))
df_churn['churn'] = (prob > 0.5).astype(int)
df_churn.to_csv('/content/type3/churn_logit1.csv', index=False)

# loan_default_logit2.csv 생성
df_loan = pd.DataFrame({
    'income': np.random.randint(2000000, 10000000, n),
    'debt_ratio': np.random.uniform(0.1, 1.5, n),
    'late_cnt': np.random.randint(0, 10, n),
})
log_odds2 = -0.000322*df_loan['income'] + 2.0093*df_loan['debt_ratio'] + 0.4774*df_loan['late_cnt']
prob2 = 1 / (1 + np.exp(-log_odds2))
df_loan['default'] = (prob2 > 0.5).astype(int)
df_loan.to_csv('/content/type3/loan_default_logit2.csv', index=False)

print("파일 생성 완료!")
print(os.listdir('/content/type3'))

파일 생성 완료!
['loan_default_logit2.csv', 'churn_logit1.csv']


/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: overflow encountered in exp
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [13]:
# [실습 2] loan_default_logit2.csv 로지스틱 회귀
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.metrics import accuracy_score
from pathlib import Path

DATA_DIR = Path('/content/type3')
df = pd.read_csv(DATA_DIR / 'loan_default_logit2.csv')

print("=== 데이터 미리보기 ===")
print(df.head())

# 로지스틱 회귀 적합
model = smf.logit('default ~ income + debt_ratio + late_cnt', data=df).fit()
print("\n=== 로지스틱 회귀 요약 ===")
print(model.summary())

# 계수 / p-value / 오즈비
result_table = pd.DataFrame({
    'coef': model.params,
    'p_value': model.pvalues,
    'odds_ratio': np.exp(model.params)
})
print("\n=== 계수 / p-value / 오즈비 ===")
print(result_table)

# 적합도 지표
llf = model.llf
residual_deviance = -2 * llf
print(f"\n=== 적합도 지표 ===")
print(f"log-likelihood: {llf:.4f}")
print(f"residual deviance: {residual_deviance:.4f}")

# 분류 성능
df['pred_prob'] = model.predict(df)
df['pred_class'] = (df['pred_prob'] >= 0.5).astype(int)
acc = accuracy_score(df['default'], df['pred_class'])
err = 1 - acc
print(f"\n=== 분류 성능 ===")
print(f"accuracy : {acc:.4f}")
print(f"error rate : {err:.4f}")

# 해석 가이드
print("\n=== 해석 가이드 ===")
for var in ['income', 'debt_ratio', 'late_cnt']:
    coef = model.params[var]
    pval = model.pvalues[var]
    or_val = np.exp(coef)
    direction = '증가' if coef > 0 else '감소'
    print(f"{var}: 계수={coef:.6f}, p-value={pval:.6f}, 오즈비={or_val:.6f}")
    print(f"  -> {var}가 1단위 증가할 때 연체 odds는 약 {or_val:.6f}배, 방향은 {direction}")

=== 데이터 미리보기 ===
    income  debt_ratio  late_cnt  default
0  9918348    0.911181         2        0
1  6988023    1.027670         5        0
2  9635836    0.233366         2        0
3  9809006    1.012973         6        0
4  9138392    0.537007         6        0
         Current function value: 0.000000
         Iterations: 35

=== 로지스틱 회귀 요약 ===
                           Logit Regression Results                           
Dep. Variable:                default   No. Observations:                  500
Model:                          Logit   Df Residuals:                      496
Method:                           MLE   Df Model:                            3
Date:                Mon, 01 Jun 2026   Pseudo R-squ.:                     inf
Time:                        13:52:30   Log-Likelihood:            -2.3943e-12
converged:                      False   LL-Null:                        0.0000
Covariance Type:            nonrobust   LLR p-value:                     1.000
             

/usr/local/lib/python3.12/dist-packages/statsmodels/discrete/discrete_model.py:227: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/usr/local/lib/python3.12/dist-packages/statsmodels/discrete/discrete_model.py:227: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/usr/local/lib/python3.12/dist-packages/statsmodels/discrete/discrete_model.py:227: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/usr/local/lib/python3.12/dist-packages/statsmodels/discrete/discrete_model.py:227: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/usr/local/lib/python3.12/dist-packa